# Feature composition analysis (derived_8.3-feature-selection-1.0)

Analyzes feature family composition, coverage, and overlap for `derived_8.3`.

In [1]:
import json
import sys
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
for p in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (p / "data" / "splits").is_dir() and (p / "Modeling").is_dir():
        PROJECT_ROOT = p
        break
sys.path.insert(0, str(PROJECT_ROOT))

EXP_DIR = PROJECT_ROOT / "notebooks" / "experiment" / "derived_8.3-feature-selection-1.0"
OUT_DIR = EXP_DIR / "artifacts" / "analysis"
OUT_DIR.mkdir(parents=True, exist_ok=True)

from Modeling.Src.soilmoist_fl.Selectors.family_coverage import (
    infer_coverage_family,
    group_by_coverage_family,
)

print("Setup analysis complete. OUT_DIR =", OUT_DIR)

Setup analysis complete. OUT_DIR = C:\Users\pan\Documents\GitHub\MDR-Project\notebooks\experiment\derived_8.3-feature-selection-1.0\artifacts\analysis


## Inspect family counts across feature sets

Group selected features into structural families: satellite, hydro, static, calendar, and temporal.

In [2]:
art_root = EXP_DIR / "artifacts" / "derived_8.3"
rows = []
if art_root.exists():
    for path in sorted(art_root.glob("*/selected_features.json")):
        payload = json.loads(path.read_text())
        feats = payload["features"]
        counts = {k: len(v) for k, v in group_by_coverage_family(feats).items()}
        row = {"variant": payload["variant"], "n": len(feats)}
        for fam in ["satellite", "hydro", "static", "calendar", "temporal", "other"]:
            row[fam] = counts.get(fam, 0)
        rows.append(row)

family_df = pd.DataFrame(rows)
if len(family_df):
    family_df.to_csv(OUT_DIR / "family_counts.csv", index=False)
    print(family_df.to_string(index=False))
else:
    print("No selection artifacts found yet.")

               variant  n  satellite  hydro  static  calendar  temporal  other
 c0_baseline_bypass_on 14          7      5       1         1         0      0
c1_baseline_bypass_off  9          6      2       1         0         0      0
                c2_xgb 50         26      7      10         7         0      0
      c2b_xgb_softcorr 55         25     12      11         7         0      0
        c2c_xgb_nocorr 55         24     12      12         7         0      0
  c2d_xgb_softcorr_k65 65         35     13      10         7         0      0
    c3_xgb_no_coverage 50         26      6      11         7         0      0
             c4_hybrid 50         26      6      11         7         0      0
                 c5_rf 50         26      7      12         5         0      0
